# 01 · Ingestion — WHO Life Expectancy (2000–2015)

**Stage 1 of the Tier A pipeline** (`DOCS/STRUCTURE.md`). Acquire the raw file, prove what it is,
and record what is wrong with it *before* anyone cleans anything.

This notebook does four things:

1. Loads the file **exactly as delivered** and preserves an immutable copy with a SHA-256 hash.
2. Validates the schema against a written contract — structure, panel balance, plausibility ranges,
   internal consistency.
3. Runs a **digit-integrity audit**, which is where this project's headline finding comes from.
4. Maps missingness three ways — by column, by country, and by *country × column with the
   all-16-years pattern flagged* — because that last cut is what exposes a merge bug.

> **Governing question for the whole project:** *Can this WHO/UN panel identify which factors a
> country could move to raise life expectancy — or does it only rank countries that already differ?*

In [1]:
import hashlib, json, sys, warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)

# --- locate the project root without hardcoding an absolute path -------------------------------
ROOT = Path.cwd()
while not (ROOT / "Life Expectancy Data.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "Life Expectancy Data.csv"
RAW, PROC, REPORTS = ROOT / "data" / "raw", ROOT / "data" / "processed", ROOT / "reports"
for p in (RAW, PROC, REPORTS):
    p.mkdir(parents=True, exist_ok=True)

print("python      :", sys.version.split()[0])
print("pandas      :", pd.__version__, "| numpy:", np.__version__)
print("project root:", ROOT.name)
print("source file :", SRC.name, "| exists:", SRC.exists())

python      : 3.14.6
pandas      : 3.0.3 | numpy: 2.5.1
project root: Life Expectancy
source file : Life Expectancy Data.csv | exists: True


## 1.1 Load exactly as delivered

Two loads are performed. The typed load is what the analysis uses; the **string load keeps every
value's stored text** (`"3488.4"`, not `3488.4`), which the digit audit in §1.5 depends on — once a
value becomes a float, its original precision is a guess.

The encoding is worth stating rather than defaulting into. The file is **UTF-8**, and
`Côte d'Ivoire` is its one non-ASCII country name. Opening it with a legacy 8-bit codec — `latin-1`
or `cp1252`, the reflexive fix on Windows — turns that name into `CÃ´te d'Ivoire`, which then fails
to match any reference table and quietly becomes a 184th country in a `groupby`.

In [2]:
df_raw = pd.read_csv(SRC, encoding="utf-8")                     # typed
df_str = pd.read_csv(SRC, encoding="utf-8", dtype=str,          # stored text, for the digit audit
                     keep_default_na=False, na_values=[""])

print(f"shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns\n")
print("columns as delivered (repr exposes stray whitespace):")
for c in df_raw.columns:
    flag = "  <-- whitespace" if c != c.strip() else ""
    print(f"   {repr(c):42s} {str(df_raw[c].dtype):8s}{flag}")

shape: 2,938 rows x 22 columns

columns as delivered (repr exposes stray whitespace):
   'Country'                                  str     
   'Year'                                     int64   
   'Status'                                   str     
   'Life expectancy '                         float64   <-- whitespace
   'Adult Mortality'                          float64 
   'infant deaths'                            int64   
   'Alcohol'                                  float64 
   'percentage expenditure'                   float64 
   'Hepatitis B'                              float64 
   'Measles '                                 int64     <-- whitespace
   ' BMI '                                    float64   <-- whitespace
   'under-five deaths '                       int64     <-- whitespace
   'Polio'                                    float64 
   'Total expenditure'                        float64 
   'Diphtheria '                              float64   <-- whitespace
   ' HIV/

### Defect 8 — eight column names carry stray whitespace

`'Life expectancy '`, `' BMI '`, `' HIV/AIDS'`, `' thinness  1-19 years'` (a *double* space).
Trivial, but it is a silent `KeyError` factory and a failed-join factory. Stripped once, here, and
logged so the change is visible rather than assumed.

In [3]:
renames = {c: c.strip() for c in df_raw.columns if c != c.strip()}
print(f"{len(renames)} column names stripped:")
for old, new in renames.items():
    print(f"   {repr(old):30s} -> {repr(new)}")

df_raw = df_raw.rename(columns=renames)
df_str = df_str.rename(columns=renames)

TARGET = "Life expectancy"
KEYS = ["Country", "Year"]
NUMERIC = [c for c in df_raw.columns if c not in ("Country", "Status", "Year")]  # Year is a key
print(f"\ntarget: {TARGET!r} | keys: {KEYS} | {len(NUMERIC)} numeric measures")

8 column names stripped:
   'Life expectancy '             -> 'Life expectancy'
   'Measles '                     -> 'Measles'
   ' BMI '                        -> 'BMI'
   'under-five deaths '           -> 'under-five deaths'
   'Diphtheria '                  -> 'Diphtheria'
   ' HIV/AIDS'                    -> 'HIV/AIDS'
   ' thinness  1-19 years'        -> 'thinness  1-19 years'
   ' thinness 5-9 years'          -> 'thinness 5-9 years'

target: 'Life expectancy' | keys: ['Country', 'Year'] | 19 numeric measures


## 1.2 Immutability & lineage

In [4]:
sha = hashlib.sha256(SRC.read_bytes()).hexdigest()
raw_copy = RAW / "Life_Expectancy_Data_raw_immutable.csv"
raw_copy.write_bytes(SRC.read_bytes())

lineage = pd.DataFrame([{
    "source_file": SRC.name,
    "sha256": sha,
    "bytes": SRC.stat().st_size,
    "pulled_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "rows": len(df_raw),
    "cols": df_raw.shape[1],
    "encoding": "utf-8",
    "provenance": "WHO Global Health Observatory + UN economic data -> Kaggle merge "
                  "(KumarRajarshi, 2018) -> this file",
    "immutable_copy": raw_copy.name,
}])
lineage.to_csv(RAW / "_lineage.csv", index=False)
print(lineage.T.to_string(header=False))

source_file                                                                                Life Expectancy Data.csv
sha256                                             872125dd1dd0f9140fbead61df20585a815f5cf47db68f08bf54efaf87963b11
bytes                                                                                                        333442
pulled_utc                                                                                2026-07-31T14:46:13+00:00
rows                                                                                                           2938
cols                                                                                                             22
encoding                                                                                                      utf-8
provenance      WHO Global Health Observatory + UN economic data -> Kaggle merge (KumarRajarshi, 2018) -> this file
immutable_copy                                                          

## 1.3 Schema validation

Every rule is written down, run, and recorded with its result — including the ones that **fail**.
A failing rule here is not a bug in the notebook; it is the finding.

In [5]:
rules = []
def check(rule, category, passed, detail=""):
    rules.append({"rule": rule, "category": category,
                  "result": "PASS" if passed else "FAIL", "detail": detail})

# --- structure ---------------------------------------------------------------------------------
dup_keys = df_raw.duplicated(KEYS).sum()
check("(Country, Year) is unique", "structure", dup_keys == 0, f"{dup_keys} duplicate keys")
dup_rows = df_raw.duplicated().sum()
check("no fully duplicated rows", "structure", dup_rows == 0, f"{dup_rows} duplicate rows")
yr_ok = df_raw.Year.between(2000, 2015).all()
check("Year in [2000, 2015]", "structure", yr_ok,
      f"observed {df_raw.Year.min()}-{df_raw.Year.max()}")
st_ok = set(df_raw.Status.dropna().unique()) <= {"Developed", "Developing"}
check("Status in {Developed, Developing}", "structure", st_ok,
      f"observed {sorted(df_raw.Status.dropna().unique())}")
st_inv = (df_raw.groupby("Country").Status.nunique() > 1).sum()
check("Status is time-invariant per country", "structure", st_inv == 0,
      f"{st_inv} countries change status")

# --- panel balance -----------------------------------------------------------------------------
per_country = df_raw.groupby("Country").Year.nunique()
balanced = (per_country == 16).all()
check("every country has 16 years", "panel", balanced,
      f"rows-per-country values: {sorted(per_country.unique())}")

# --- plausibility ranges -----------------------------------------------------------------------
ranges = {
    TARGET: (20, 90), "Adult Mortality": (0, 1000), "Hepatitis B": (0, 100),
    "Polio": (0, 100), "Diphtheria": (0, 100), "BMI": (10, 50),
    "Income composition of resources": (0, 1), "Schooling": (0, 25),
    "Total expenditure": (0, 100), "percentage expenditure": (0, 100),
}
for col, (lo, hi) in ranges.items():
    s = df_raw[col].dropna()
    bad = ((s < lo) | (s > hi)).sum()
    check(f"{col} in [{lo}, {hi}]", "plausibility", bad == 0,
          f"{bad} violations; observed {s.min():g}-{s.max():g}")

# --- non-negativity ----------------------------------------------------------------------------
for col in ["infant deaths", "under-five deaths", "Measles", "GDP", "Population", "HIV/AIDS",
            "Alcohol", "thinness  1-19 years", "thinness 5-9 years"]:
    bad = (df_raw[col].dropna() < 0).sum()
    check(f"{col} >= 0", "plausibility", bad == 0, f"{bad} negative values")

# --- internal consistency ----------------------------------------------------------------------
viol = (df_raw["under-five deaths"] < df_raw["infant deaths"]).sum()
check("under-five deaths >= infant deaths", "consistency", viol == 0, f"{viol} violations")
r = df_raw[["infant deaths", "under-five deaths"]].corr().iloc[0, 1]
check("infant vs under-five deaths are not near-duplicates", "consistency", r < 0.95,
      f"Pearson r = {r:.4f}")

schema = pd.DataFrame(rules)
schema.to_csv(RAW / "_schema_validation.csv", index=False)
print(schema.to_string(index=False))
print(f"\n{(schema.result=='PASS').sum()} passed / {(schema.result=='FAIL').sum()} FAILED")

                                               rule     category result                                               detail
                          (Country, Year) is unique    structure   PASS                                     0 duplicate keys
                           no fully duplicated rows    structure   PASS                                     0 duplicate rows
                               Year in [2000, 2015]    structure   PASS                                   observed 2000-2015
                  Status in {Developed, Developing}    structure   PASS                 observed ['Developed', 'Developing']
               Status is time-invariant per country    structure   PASS                            0 countries change status
                         every country has 16 years        panel   FAIL rows-per-country values: [np.int64(1), np.int64(16)]
                        Life expectancy in [20, 90] plausibility   PASS                       0 violations; observed 36.3-89


**Six rules fail, and each is a real defect rather than a mis-specified rule:**

| Failing rule | What it means |
|---|---|
| `every country has 16 years` | the panel is **not balanced** — investigated in §1.4 |
| `BMI in [10, 50]` | values run **1.0 → 87.3**; no national mean BMI is 87 |
| `percentage expenditure in [0, 100]` | reaches **19,480** — it is not a percentage |
| `Total expenditure in [0,100]` / `Income composition in [0,1]` | pass, but see the zero-coding in notebook 02 |
| `under-five vs infant deaths not near-duplicates` | r = **0.997** — two columns carrying one signal |

`Life expectancy in [20, 90]` **passes** — which is exactly the problem. Its maximum, 89.0, is
inside any sane range and is still wrong (§1.5).

## 1.4 Panel balance — "193 countries × 16 years" is false

The analysis brief describes this as a panel of the same 193 countries tracked across 16 years.
It is not.

In [6]:
counts = df_raw.groupby("Country").Year.nunique().value_counts().sort_index()
print("countries by number of years present:")
for n_years, n_countries in counts.items():
    print(f"   {n_years:2d} years : {n_countries:3d} countries")

singletons = df_raw.groupby("Country").Year.nunique().pipe(lambda s: s[s < 16]).index.tolist()
print(f"\n{len(singletons)} single-row countries:")
sing = df_raw[df_raw.Country.isin(singletons)][["Country", "Year", "Status", TARGET, "GDP", "Schooling"]]
print(sing.to_string(index=False))

print(f"\nreconciliation: 183 x 16 = {183*16:,}  +  {len(singletons)} singletons  =  {183*16+len(singletons):,} rows")
print(f"target null in singletons: {sing[TARGET].isna().sum()} of {len(sing)}")
print(f"target null overall      : {df_raw[TARGET].isna().sum()}")

countries by number of years present:
    1 years :  10 countries
   16 years : 183 countries

10 single-row countries:
              Country  Year     Status  Life expectancy         GDP  Schooling
         Cook Islands  2013 Developing              NaN         NaN        NaN
             Dominica  2013 Developing              NaN  722.756650       12.7
     Marshall Islands  2013 Developing              NaN 3617.752354        0.0
               Monaco  2013 Developing              NaN         NaN        NaN
                Nauru  2013 Developing              NaN  136.183210        9.6
                 Niue  2013 Developing              NaN         NaN        NaN
                Palau  2013 Developing              NaN 1932.122370       14.2
Saint Kitts and Nevis  2013 Developing              NaN         NaN       13.4
           San Marino  2013 Developing              NaN         NaN       15.1
               Tuvalu  2013 Developing              NaN 3542.135890        0.0

reconcilia

**So What:** the ten "extra" countries contribute a single 2013 row each, and **all ten have a null
life expectancy** — they are exactly the ten missing-target rows in the file. They cannot enter a
panel model or a supervised fit. They are removed in notebook 02, by name and with a logged reason,
and the row count reconciles 2,938 → 2,928.

**Implication:** any analysis that reports "193 countries" is counting ten countries that carry no
outcome data.

## 1.5 The digit-integrity audit

This is the project's headline check, and it is ten lines of code.

**The idea.** In any naturally-occurring numeric column, the digits *after* the leading one are
close to uniformly distributed — leading digits follow Benford's law, but trailing digits do not.
So roughly 1 in 10 of those positions should be a `0`, and a value with *k* trailing digits should
contain at least one `0` with probability `1 − 0.9^k`.

**The test.** For each column, compare the observed share of values containing a `0` against that
expectation. A column where the observed share is **exactly zero** has not been rounded, scaled, or
sampled — it has been *edited*.

In [7]:
from scipy import stats

def digit_string(token: str) -> str:
    # Digits of a stored value, leading zeros removed ('0.1' -> '1', '3488.4' -> '34884')
    return token.replace(".", "").replace("-", "").lstrip("0")

def digit_audit(str_frame: pd.DataFrame, cols) -> pd.DataFrame:
    rows = []
    for c in cols:
        toks = str_frame[c].dropna()
        toks = toks[toks.str.strip() != ""]
        digits = toks.map(digit_string)
        digits = digits[digits.str.len() > 0]
        k = digits.str.len() - 1                        # digits after the leading one
        has0 = digits.str.slice(1).str.contains("0")
        p_expected = (1 - 0.9 ** k)                     # P(at least one 0) under a uniform null
        # chi-square on the pooled non-leading digit frequencies
        pool = "".join(digits.str.slice(1))
        freq = pd.Series(list(pool)).value_counts().reindex(list("0123456789"), fill_value=0)
        chi = stats.chisquare(freq.values) if freq.sum() > 0 else None
        rows.append({
            "column": c,
            "n_values": len(digits),
            "mean_digits": round(digits.str.len().mean(), 2),
            "obs_share_with_0": round(has0.mean(), 4),
            "exp_share_with_0": round(p_expected.mean(), 4),
            "n_values_with_0": int(has0.sum()),
            "zeros_per_1000_digits": round(1000 * freq["0"] / max(freq.sum(), 1), 1),
            "chi2": round(chi.statistic, 1) if chi else np.nan,
            "chi2_p": chi.pvalue if chi else np.nan,
        })
    out = pd.DataFrame(rows)
    out["shortfall_rows"] = (out.exp_share_with_0 * out.n_values - out.n_values_with_0).round().astype(int)
    out["verdict"] = np.where(out.n_values_with_0 == 0, "CORRUPT (no 0 digits at all)",
                       np.where(out.obs_share_with_0 < 0.5 * out.exp_share_with_0,
                                "SUSPECT (0 digits depleted)", "ok"))
    return out.sort_values("shortfall_rows", ascending=False)

audit = digit_audit(df_str, NUMERIC)
audit.to_csv(PROC / "_digit_audit.csv", index=False)
print(audit.drop(columns=["chi2", "chi2_p"]).to_string(index=False))

                         column  n_values  mean_digits  obs_share_with_0  exp_share_with_0  n_values_with_0  zeros_per_1000_digits  shortfall_rows                      verdict
                            GDP      2490         9.07            0.0000            0.5708                0                    0.0            1421 CORRUPT (no 0 digits at all)
                     Population      2286         6.53            0.0000            0.4370                0                    0.0             999 CORRUPT (no 0 digits at all)
                            BMI      2904         2.82            0.0000            0.1735                0                    0.0             504 CORRUPT (no 0 digits at all)
              Total expenditure      2712         2.84            0.0000            0.1751                0                    0.0             475 CORRUPT (no 0 digits at all)
                Life expectancy      2928         2.80            0.0099            0.1719               29             

In [8]:
corrupt = audit[audit.verdict.str.startswith("CORRUPT")]
suspect = audit[audit.verdict.str.startswith("SUSPECT")]
print(f"CORRUPT columns : {len(corrupt)}  -> {list(corrupt.column)}\n")
print(f"SUSPECT columns : {len(suspect)}  -> {list(suspect.column)}\n")
print(f"estimated corrupted cells: {audit.shortfall_rows.clip(lower=0).sum():,} "
      f"of {df_raw[NUMERIC].notna().sum().sum():,} non-null numeric cells "
      f"({100*audit.shortfall_rows.clip(lower=0).sum()/df_raw[NUMERIC].notna().sum().sum():.1f}%)\n")
for _, r in corrupt.iterrows():
    print(f"  {r.column:32s} 0 of {r.n_values:5,} values contain a '0' "
          f"-- {100*r.exp_share_with_0:5.1f}% expected  (chi2 p = {r.chi2_p:.3g})")

CORRUPT columns : 10  -> ['GDP', 'Population', 'BMI', 'Total expenditure', 'Adult Mortality', 'Polio', 'Diphtheria', 'thinness 5-9 years', 'thinness  1-19 years', 'Hepatitis B']

SUSPECT columns : 2  -> ['Life expectancy', 'HIV/AIDS']

estimated corrupted cells: 6,217 of 53,259 non-null numeric cells (11.7%)

  GDP                              0 of 2,490 values contain a '0' --  57.1% expected  (chi2 p = 0)
  Population                       0 of 2,286 values contain a '0' --  43.7% expected  (chi2 p = 1.07e-299)
  BMI                              0 of 2,904 values contain a '0' --  17.3% expected  (chi2 p = 5.86e-122)
  Total expenditure                0 of 2,712 values contain a '0' --  17.5% expected  (chi2 p = 1.79e-117)
  Adult Mortality                  0 of 2,928 values contain a '0' --  15.4% expected  (chi2 p = 8.3e-113)
  Polio                            0 of 2,919 values contain a '0' --   9.4% expected  (chi2 p = 2.31e-130)
  Diphtheria                       0 of 2,919 valu

### The digit `0` has been deleted from ten columns

Ten columns contain **not a single `0` digit** where 8.6%–57.1% of their values should have one.
`GDP` is the clearest: 0 of 2,490 values, against 57.1% expected. Under the uniform null the
probability of that is below 1e-500.

Three independent lines of evidence, so the claim does not rest on one statistic:

In [9]:
print("EVIDENCE 1 - positional: Adult Mortality's last digit is never 0\n")
am = df_str["Adult Mortality"].dropna()
last = am.str.replace(".0", "", regex=False).str[-1]
tab = last.value_counts().sort_index()
print(pd.DataFrame({"count": tab, "pct": (100 * tab / tab.sum()).round(1)}).to_string())
print(f"\n   length distribution: {am.str.replace('.0','',regex=False).str.len().value_counts().sort_index().to_dict()}")
print("   -> 86 single-digit adult-mortality rates in a column whose median is 144.")

EVIDENCE 1 - positional: Adult Mortality's last digit is never 0

                 count   pct
Adult Mortality             
1                  306  10.5
2                  327  11.2
3                  303  10.3
4                  342  11.7
5                  304  10.4
6                  332  11.3
7                  335  11.4
8                  361  12.3
9                  318  10.9

   length distribution: {1: 86, 2: 982, 3: 1860}
   -> 86 single-digit adult-mortality rates in a column whose median is 144.


In [10]:
print("EVIDENCE 2 - reconstructive: sorted by year, the damage is self-evident\n")
for country in ["Saudi Arabia", "Belgium", "Barbados"]:
    s = (df_raw[df_raw.Country == country].sort_values("Year")["Adult Mortality"]
         .astype("Int64").tolist())
    print(f"   {country:14s} {s}")
print("\n   Saudi Arabia reads as 110,107,105,103,102,101,100,99,98,97,94,92,90,89,88,88")
print("   once the deleted zeros are restored - a monotone decline. Belgium's '8' is 80.")

EVIDENCE 2 - reconstructive: sorted by year, the damage is self-evident

   Saudi Arabia   [11, 17, 15, 13, 12, 11, 1, 99, 98, 97, 94, 92, 9, 89, 88, 88]
   Belgium        [11, 1, 99, 95, 91, 89, 85, 86, 87, 85, 81, 8, 78, 77, 76, 74]
   Barbados       [127, 125, 123, 121, 119, 117, 115, 113, 111, 19, 16, 14, 12, 11, 1, 98]

   Saudi Arabia reads as 110,107,105,103,102,101,100,99,98,97,94,92,90,89,88,88
   once the deleted zeros are restored - a monotone decline. Belgium's '8' is 80.


In [11]:
print("EVIDENCE 3 - external: World Bank GDP per capita, delete the '0', get the file's value\n")
truth = {("Japan", 2003): 34808, ("Japan", 2009): 40855, ("Japan", 2010): 44508,
         ("Japan", 2013): 40454, ("Norway", 2002): 43061, ("India", 2009): 1090}
rows = []
for (c, y), t in truth.items():
    got = df_raw.loc[(df_raw.Country == c) & (df_raw.Year == y), "GDP"].iloc[0]
    stripped = int(str(t).replace("0", ""))
    rows.append({"country": c, "year": y, "file_value": round(got, 1),
                 "world_bank": t, "wb_with_0s_deleted": stripped,
                 "match": abs(got - stripped) < 1})
print(pd.DataFrame(rows).to_string(index=False))

EVIDENCE 3 - external: World Bank GDP per capita, delete the '0', get the file's value

country  year  file_value  world_bank  wb_with_0s_deleted  match
  Japan  2003      3488.4       34808                3488   True
  Japan  2009      4855.2       40855                4855   True
  Japan  2010      4457.7       44508                4458   True
  Japan  2013      4454.4       40454                4454   True
 Norway  2002      4361.2       43061                4361   True
  India  2009        19.3        1090                  19   True


### The target is corrupted too — `80.9` was stored as `89.0`

`Life expectancy` is stored to one decimal, so deleting the `0` from `80.9` leaves `89` and the
value is re-read at the original precision as `89.0`. That predicts a very specific signature: the
**units** digit should be depleted of `0`s and the **decimal** digit should carry a matching excess.

In [12]:
le = df_raw[TARGET].dropna()
s = le.map(lambda x: f"{x:.1f}")
units, dec = s.str[-3], s.str[-1]
n = len(s)

pos = pd.DataFrame({"units_digit": units.value_counts().sort_index(),
                    "decimal_digit": dec.value_counts().sort_index()}).fillna(0).astype(int)
pos["units_pct"] = (100 * pos.units_digit / n).round(1)
pos["decimal_pct"] = (100 * pos.decimal_digit / n).round(1)
print(pos.to_string())

chi_u = stats.chisquare(units.value_counts().sort_index().values)
chi_d = stats.chisquare(dec.value_counts().sort_index().values)
print(f"\n   units digit '0' : {(units=='0').sum():4d} observed vs {n*0.1:5.0f} expected "
      f"({100*(units=='0').mean():.1f}% vs 10%)   chi2 p = {chi_u.pvalue:.2e}")
print(f"   decimal digit '0': {(dec=='0').sum():4d} observed vs {n*0.1:5.0f} expected "
      f"({100*(dec=='0').mean():.1f}% vs 10%)   chi2 p = {chi_d.pvalue:.2e}")
print(f"\n   depletion at units=0 : {n*0.1 - (units=='0').sum():+.0f} rows")
print(f"   excess at decimal=0  : {(dec=='0').sum() - n*0.1:+.0f} rows   <- they match")

                 units_digit  decimal_digit  units_pct  decimal_pct
Life expectancy                                                    
0                         29            589        1.0         20.1
1                        337            257       11.5          8.8
2                        355            243       12.1          8.3
3                        370            263       12.6          9.0
4                        346            263       11.8          9.0
5                        312            266       10.7          9.1
6                        268            263        9.2          9.0
7                        268            274        9.2          9.4
8                        325            264       11.1          9.0
9                        318            246       10.9          8.4

   units digit '0' :   29 observed vs   293 expected (1.0% vs 10%)   chi2 p = 4.81e-59
   decimal digit '0':  589 observed vs   293 expected (20.1% vs 10%)   chi2 p = 7.32e-67

   dep

In [13]:
print("The file's maximum life expectancy is a typo:\n")
mx = df_raw.nlargest(4, TARGET)[["Country", "Year", TARGET]]
print(mx.to_string(index=False))
print("\nItaly's neighbouring years:")
it = df_raw[df_raw.Country == "Italy"].sort_values("Year")[["Year", TARGET]]
print("   ", it[TARGET].tolist())
print("\n   2004 reads 89.0 and 2005 reads 88.0, between 79.9 and 81.2.")
print("   Restoring the deleted zero gives 80.9 and 80.8 - which is the actual series.")

The file's maximum life expectancy is a typo:

Country  Year  Life expectancy
Belgium  2014             89.0
Finland  2014             89.0
 France  2008             89.0
 France  2007             89.0

Italy's neighbouring years:
    [79.4, 79.8, 80.0, 79.9, 89.0, 88.0, 81.2, 81.3, 81.5, 81.6, 81.8, 82.0, 82.0, 82.3, 82.5, 82.7]

   2004 reads 89.0 and 2005 reads 88.0, between 79.9 and 81.2.
   Restoring the deleted zero gives 80.9 and 80.8 - which is the actual series.


**So What:** roughly one life-expectancy value in ten is wrong, in a column that passes every range
check, every null check and every dtype check. No amount of missing-value strategy or outlier
trimming would have found this — only the digit audit does.

**Implication:** every model ever fitted on this file has been fitted partly to fiction, and the
repair is testable rather than speculative (notebook 02).

## 1.6 Missingness — three cuts, because only the third one is diagnostic

In [14]:
miss_col = pd.DataFrame({"n_missing": df_raw.isna().sum(),
                         "pct": (100 * df_raw.isna().mean()).round(2)})
miss_col = miss_col[miss_col.n_missing > 0].sort_values("n_missing", ascending=False)
print("BY COLUMN (reproduces the analysis brief's table exactly):")
print(miss_col.to_string())

BY COLUMN (reproduces the analysis brief's table exactly):
                                 n_missing    pct
Population                             652  22.19
Hepatitis B                            553  18.82
GDP                                    448  15.25
Total expenditure                      226   7.69
Alcohol                                194   6.60
Income composition of resources        167   5.68
Schooling                              163   5.55
thinness 5-9 years                      34   1.16
thinness  1-19 years                    34   1.16
BMI                                     34   1.16
Polio                                   19   0.65
Diphtheria                              19   0.65
Life expectancy                         10   0.34
Adult Mortality                         10   0.34


In [15]:
miss_country = df_raw.set_index("Country").isna().sum(axis=1).groupby(level=0).sum().sort_values(ascending=False)
print("BY COUNTRY (top 12):")
print(miss_country.head(12).to_string())
print("\n   The brief flags this as counter-intuitive: the UK, Czechia, the USA and the Republic of")
print("   Korea sit beside South Sudan and Somalia. The next cut explains why.")

BY COUNTRY (top 12):
Country
South Sudan                                             122
Somalia                                                  91
Democratic People's Republic of Korea                    84
United Kingdom of Great Britain and Northern Ireland     81
Democratic Republic of the Congo                         73
United Republic of Tanzania                              68
Czechia                                                  68
Côte d'Ivoire                                            67
United States of America                                 66
Republic of Korea                                        66
Republic of Moldova                                      66
Sudan                                                    55

   The brief flags this as counter-intuitive: the UK, Czechia, the USA and the Republic of
   Korea sit beside South Sudan and Somalia. The next cut explains why.


In [16]:
# the diagnostic cut: which countries are missing a column for EVERY year they appear?
panel_countries = df_raw.groupby("Country").Year.nunique().pipe(lambda s: s[s == 16]).index
sub = df_raw[df_raw.Country.isin(panel_countries)]
allmiss = sub.groupby("Country")[NUMERIC].apply(lambda g: g.isna().all()).sum().sort_values(ascending=False)
print("countries missing a column for ALL 16 years:")
print(allmiss[allmiss > 0].to_string())

gdp_gone = sorted(sub.groupby("Country").GDP.apply(lambda s: s.isna().all()).pipe(lambda x: x[x]).index)
pop_gone = sorted(sub.groupby("Country").Population.apply(lambda s: s.isna().all()).pipe(lambda x: x[x]).index)
print(f"\nGDP missing all 16 years -- {len(gdp_gone)} countries:")
for c in gdp_gone:
    print("   ", c)
print(f"\nPopulation missing all 16 years -- {len(pop_gone)} countries")
print(f"every GDP-missing country is also Population-missing: {set(gdp_gone) <= set(pop_gone)}")

countries missing a column for ALL 16 years:


Population                         40
GDP                                25
Schooling                          10
Income composition of resources    10
Hepatitis B                         9
Total expenditure                   2
BMI                                 2
thinness  1-19 years                2
thinness 5-9 years                  2
Alcohol                             1

GDP missing all 16 years -- 25 countries:
    Bahamas
    Bolivia (Plurinational State of)
    Congo
    Czechia
    Côte d'Ivoire
    Democratic People's Republic of Korea
    Democratic Republic of the Congo
    Egypt
    Gambia
    Iran (Islamic Republic of)
    Kyrgyzstan
    Lao People's Democratic Republic
    Micronesia (Federated States of)
    Republic of Korea
    Republic of Moldova
    Saint Lucia
    Saint Vincent and the Grenadines
    Slovakia
    The former Yugoslav republic of Macedonia
    United Kingdom of Great Britain and Northern Ireland
    United Republic of Tanzania
    United States of

### This is a merge bug, not a reporting-capacity story

The list is not "fragile states". It is `United Kingdom of Great Britain and Northern Ireland`,
`United States of America`, `Republic of Korea`, `Czechia`, `Viet Nam`, `Egypt`, `Slovakia`,
`Bolivia (Plurinational State of)`, `Iran (Islamic Republic of)`, `Côte d'Ivoire` — **WHO formal
names that do not match World Bank / UN keys** (`United Kingdom`, `United States`, `Korea, Rep.`,
`Czech Republic`, `Vietnam`, `Egypt, Arab Rep.`, `Slovak Republic`, `Bolivia`, `Iran, Islamic Rep.`).

The missingness is *structural and total per country* — 16 of 16 years — which is the signature of a
failed join key, not of a country that reports irregularly.

**So What:** the brief attributes this to "differing WHO/UNSD reporting schedules". It is a merge
key. That matters because a schedule gap is MAR and imputable, while a failed join is MNAR and
**not** imputable from within the file — you cannot invent the United States' GDP from Chad's.

**Implication:** `dropna()` on this file silently deletes the United States, Japan and Denmark.
Quantified in notebook 05's representation audit.

In [17]:
miss_summary = pd.concat([
    miss_col.assign(cut="by_column").reset_index(names="key"),
    miss_country.rename("n_missing").to_frame().assign(cut="by_country").reset_index(names="key"),
], ignore_index=True)
miss_summary.to_csv(REPORTS / "_missingness.csv", index=False)

pd.DataFrame({"country": gdp_gone, "gdp_missing_all_16y": True}).to_csv(
    REPORTS / "_join_failures.csv", index=False)
print(f"wrote _missingness.csv ({len(miss_summary)} rows) and _join_failures.csv ({len(gdp_gone)} rows)")

wrote _missingness.csv (207 rows) and _join_failures.csv (25 rows)


## 1.7 Governance

In [18]:
gov = pd.DataFrame([{
    "data_class": "public / open",
    "pii": "none - country-year aggregates, no individuals",
    "source_licence": "WHO GHO + UN, public domain; Kaggle redistribution",
    "sensitive_attributes": "none at person level; country development status present",
    "person_level_fairness_audit": "NOT APPLICABLE - no individual is scored",
    "country_level_representation_audit": "REQUIRED - scheduled in notebook 05 (complete-case bias)",
    "safe_to_commit": "yes - notebooks + report only; raw data stays local",
    "retention": "source retained locally under data/raw/, immutable",
}])
gov.T.to_csv(RAW / "_governance.csv", header=False)
print(gov.T.to_string(header=False))

data_class                                                                     public / open
pii                                           none - country-year aggregates, no individuals
source_licence                            WHO GHO + UN, public domain; Kaggle redistribution
sensitive_attributes                none at person level; country development status present
person_level_fairness_audit                         NOT APPLICABLE - no individual is scored
country_level_representation_audit  REQUIRED - scheduled in notebook 05 (complete-case bias)
safe_to_commit                           yes - notebooks + report only; raw data stays local
retention                                 source retained locally under data/raw/, immutable


## Stage 1 — Gate Checklist

- [x] **Raw data saved to `data/raw/`** (untouched copy + SHA-256 hash)
- [x] **Source, timestamp, and row/column counts logged** — `_lineage.csv`, provenance chain recorded
- [x] **Schema matches expectations, or discrepancies are documented** — 6 rules FAIL and each is
      carried forward as a named defect rather than silently patched
- [x] **PII / data governance review completed** — `_governance.csv`; no person-level data, so the
      person-level fairness audit is recorded as *not applicable with reason*, and the
      country-level representation audit is scheduled rather than skipped

**Additional evidence produced at this stage (beyond the standard gate):**

- [x] **Digit-integrity audit** — 10 columns CORRUPT, ~6,100 cells affected, `_digit_audit.csv`
- [x] **Target corruption proven** — units/decimal digit signature, χ² p ≈ 1e-59 and 1e-67
- [x] **Panel balance checked** — 183 × 16 + 10 singletons, all 10 with a null target
- [x] **Missingness mapped three ways** — the country × all-years cut exposes 25 join failures

### Changelog

| # | Change | Rows/cols affected |
|---|---|---|
| 1 | Stripped whitespace from 8 column names | 8 columns |
| 2 | Nothing else — this stage is read-only by design | 0 rows |

**Next:** `02_cleaning.ipynb` — the eight defects, the ledgered repair, and the unit contract.